In [2]:
import pandas as pd
import os
from pathlib import Path

# Get the path to final_datasets directory
data_dir = Path('../../data/final_datasets')

# Get all CSV files in the directory
csv_files = list(data_dir.glob('*.csv'))
print(f"Found {len(csv_files)} CSV files:")
for file in csv_files:
    print(f"  - {file.name}")

# Read and merge all datasets
dataframes = []
for file in csv_files:
    df = pd.read_csv(file)
    print(f"\n{file.name}: {len(df)} rows")
    dataframes.append(df)

# Concatenate all dataframes
merged_df = pd.concat(dataframes, ignore_index=True)

print(f"\nMerged dataset: {len(merged_df)} rows, {len(merged_df.columns)} columns")
print(f"\nFirst few rows:")
merged_df.head()


Found 2 CSV files:
  - 28k_new_products_with_images.csv
  - 40k_products_with_images.csv

28k_new_products_with_images.csv: 28058 rows

40k_products_with_images.csv: 36518 rows

Merged dataset: 64576 rows, 65 columns

First few rows:


,Unnamed: 0,asin,category,query,page,source_section,type,title,image,has_prime,...,sd_ratings_distribution,sd_customer_sentiments,sd_error,sd_errors,sd_rating_pct_1,sd_rating_pct_2,sd_rating_pct_3,sd_rating_pct_4,sd_rating_pct_5,Unnamed: 0.1
0,0,B0D9VR51R3,home_kitchen,kitchen gadget,3,results,search_product,"Silicone Flexible Cooking Fork, 11.6 Inch Heat...",https://m.media-amazon.com/images/I/61LRx4da3u...,False,...,"[{""rating"": 5, ""distribution"": ""79""}, {""rating...",[],NaN,NaN,1.0,1.0,6.0,13.0,79.0,NaN
1,1,B0D5H6GFVJ,home_kitchen,kitchen gadget,3,results,search_product,"FLAIROSOL OLIVIA Oil Sprayer for Cooking, 200m...",https://m.media-amazon.com/images/I/5108uzHo+V...,False,...,"[{""rating"": 5, ""distribution"": ""80""}, {""rating...",[],NaN,NaN,4.0,2.0,4.0,10.0,80.0,NaN
2,2,B0CY1S56JZ,home_kitchen,kitchen gadget,3,results,search_product,TrendPlain 16oz/470ml Glass Olive Oil Sprayer ...,https://m.media-amazon.com/images/I/71++rikV+5...,False,...,"[{""rating"": 5, ""distribution"": ""77""}, {""rating...",[],NaN,NaN,5.0,2.0,4.0,12.0,77.0,NaN
3,3,B092DBTWCF,home_kitchen,kitchen gadget,3,results,search_product,ADBIU Over The Sink Dish Drying Rack (Expandab...,https://m.media-amazon.com/images/I/91e9eDAfn5...,False,...,"[{""rating"": 5, ""distribution"": ""73""}, {""rating...",[],NaN,NaN,5.0,2.0,5.0,15.0,73.0,NaN
4,4,B0C5HTS85W,home_kitchen,kitchen gadget,3,results,search_product,Elite Gourmet EGC115M Easy Egg Cooker Electric...,https://m.media-amazon.com/images/I/61PCEgXrOX...,False,...,"[{""rating"": 5, ""distribution"": ""79""}, {""rating...",[],NaN,NaN,4.0,2.0,4.0,11.0,79.0,NaN


In [5]:
merged_df.columns

Index(['Unnamed: 0', 'asin', 'category', 'query', 'page', 'source_section',
       'type', 'title', 'image', 'has_prime', 'is_best_seller',
       'is_amazon_choice', 'limited_time_deal', 'deal_of_the_day', 'stars',
       'total_reviews', 'url', 'optimized_url', 'sponsored',
       'number_of_people_bought', 'delivery', 'availability_quantity',
       'price_string', 'price_symbol', 'price', 'absolute_position',
       'organic_position', 'certification', 'coupon_text', 'colors',
       'num_colors', 'location', 'search_message', 'fetched_at_unix',
       'sd_feature_bullets_text', 'sd_title', 'sd_parent_asin', 'sd_price',
       'sd_list_price', 'sd_previous_price', 'sd_price_symbol',
       'sd_availability_status', 'sd_aplus', 'sd_is_prime_exclusive',
       'sd_is_frequently_returned', 'sd_number_bought_past_month',
       'sd_main_image', 'sd_images', 'sd_average_rating', 'sd_total_reviews',
       'sd_best_sellers_rank', 'sd_product_category', 'sd_category_id',
       'sd_rating

In [8]:
import re

def parse_bsr(bsr_string):
    """
    Parse BSR string to extract main BSR and lowest BSR.
    Format: #rank in Category (See Top 100 in Category)  #rank in Category ...
    """
    if pd.isna(bsr_string) or bsr_string == '':
        return None, None, None, None
    
    # Pattern to match: #number in Category
    # The number may contain commas
    pattern = r'#([\d,]+)\s+in\s+([^#(]+?)(?:\s*\(|$)'
    
    matches = re.findall(pattern, str(bsr_string))
    
    if not matches:
        return None, None, None, None
    
    # Parse all BSR entries
    bsr_entries = []
    for rank_str, category in matches:
        # Remove commas from rank and convert to int
        rank = int(rank_str.replace(',', ''))
        # Clean up category (remove trailing spaces)
        category = category.strip()
        bsr_entries.append((rank, category))
    
    if not bsr_entries:
        return None, None, None, None
    
    # Main BSR is the first one
    main_rank, main_category = bsr_entries[0]
    
    # Lowest BSR is the one with the smallest rank (closest to 1)
    lowest_rank, lowest_category = min(bsr_entries, key=lambda x: x[0])
    
    return main_category, main_rank, lowest_category, lowest_rank

# Apply the parsing function
print("Parsing BSR data...")
results = merged_df['sd_best_sellers_rank'].apply(parse_bsr)

# Extract the results into separate columns
merged_df['main_bsr_group'] = [r[0] if r else None for r in results]
merged_df['main_bsr_rank'] = [r[1] if r else None for r in results]
merged_df['lowest_bsr_group'] = [r[2] if r else None for r in results]
merged_df['lowest_bsr_rank'] = [r[3] if r else None for r in results]

print(f"\nBSR parsing complete!")
print(f"\nSample of parsed data:")
print(merged_df[['sd_best_sellers_rank', 'main_bsr_group', 'main_bsr_rank', 'lowest_bsr_group', 'lowest_bsr_rank']].head(10))


Parsing BSR data...

BSR parsing complete!

Sample of parsed data:
                                sd_best_sellers_rank    main_bsr_group  \
0  #11,635 in Kitchen & Dining (See Top 100 in Ki...  Kitchen & Dining   
1  #484 in Kitchen & Dining (See Top 100 in Kitch...  Kitchen & Dining   
2  #6 in Kitchen & Dining (See Top 100 in Kitchen...  Kitchen & Dining   
3  #1,481 in Kitchen & Dining (See Top 100 in Kit...  Kitchen & Dining   
4  #50 in Kitchen & Dining (See Top 100 in Kitche...  Kitchen & Dining   
5  #87 in Office Products (See Top 100 in Office ...   Office Products   
6  #438 in Kitchen & Dining (See Top 100 in Kitch...  Kitchen & Dining   
7  #3,416 in Kitchen & Dining (See Top 100 in Kit...  Kitchen & Dining   
8  #450 in Home & Kitchen (See Top 100 in Home & ...    Home & Kitchen   
9  #13,292 in Kitchen & Dining (See Top 100 in Ki...  Kitchen & Dining   

   main_bsr_rank             lowest_bsr_group  lowest_bsr_rank  
0        11635.0             Cooking Utensils        

In [9]:
merged_df.to_csv('./../../data/final_datasets/68k_multisector_amazon_products_final.csv')

In [10]:
merged_df.columns

Index(['Unnamed: 0', 'asin', 'category', 'query', 'page', 'source_section',
       'type', 'title', 'image', 'has_prime', 'is_best_seller',
       'is_amazon_choice', 'limited_time_deal', 'deal_of_the_day', 'stars',
       'total_reviews', 'url', 'optimized_url', 'sponsored',
       'number_of_people_bought', 'delivery', 'availability_quantity',
       'price_string', 'price_symbol', 'price', 'absolute_position',
       'organic_position', 'certification', 'coupon_text', 'colors',
       'num_colors', 'location', 'search_message', 'fetched_at_unix',
       'sd_feature_bullets_text', 'sd_title', 'sd_parent_asin', 'sd_price',
       'sd_list_price', 'sd_previous_price', 'sd_price_symbol',
       'sd_availability_status', 'sd_aplus', 'sd_is_prime_exclusive',
       'sd_is_frequently_returned', 'sd_number_bought_past_month',
       'sd_main_image', 'sd_images', 'sd_average_rating', 'sd_total_reviews',
       'sd_best_sellers_rank', 'sd_product_category', 'sd_category_id',
       'sd_rating

In [ ]:
scraper_df['keyword'].unique().tolist()

['audio headphones catalog full 1757641194',
 'computers catalog full 1757638340',
 'gaming catalog full 1757700673',
 'mobile phones catalog full 1757639774',
 'other catalog full 1757715295',
 'photography catalog full 1757703787',
 'smart devices catalog full 1757706192',
 'smart home catalog full 1757713912',
 'tv catalog full 1757643738']

In [11]:
# Read the scraper data
scraper_df = pd.read_csv('../../data/data_with_scraper.csv')

print(f"Scraper data: {len(scraper_df)} rows, {len(scraper_df.columns)} columns")
print(f"Merged data: {len(merged_df)} rows, {len(merged_df.columns)} columns")

# Ensure asin columns are strings and stripped
merged_df['asin'] = merged_df['asin'].astype(str).str.strip()
scraper_df['asin'] = scraper_df['asin'].astype(str).str.strip()

# Check for and remove duplicate ASINs in merged_df BEFORE merging
merged_duplicates = merged_df['asin'].duplicated().sum()
if merged_duplicates > 0:
    print(f"\n⚠️  Found {merged_duplicates} duplicate ASINs in merged_df. Removing duplicates (keeping first)...")
    merged_df = merged_df.drop_duplicates(subset=['asin'], keep='first')
    print(f"   After removal: {len(merged_df)} rows")
else:
    print(f"\n✓ No duplicate ASINs in merged_df")

# Check for and remove duplicate ASINs in scraper_df BEFORE merging
scraper_duplicates = scraper_df['asin'].duplicated().sum()
if scraper_duplicates > 0:
    print(f"⚠️  Found {scraper_duplicates} duplicate ASINs in scraper_df. Removing duplicates (keeping first)...")
    scraper_df = scraper_df.drop_duplicates(subset=['asin'], keep='first')
    print(f"   After removal: {len(scraper_df)} rows")
else:
    print(f"✓ No duplicate ASINs in scraper_df")

# Find overlapping columns (excluding 'asin')
overlapping_cols = set(merged_df.columns) & set(scraper_df.columns) - {'asin'}
print(f"\nOverlapping columns (excluding 'asin'): {len(overlapping_cols)}")
if overlapping_cols:
    print(f"  Examples: {list(overlapping_cols)[:10]}")

# Rename overlapping columns in scraper_df to avoid conflicts
# We'll keep merged_df's version, so rename scraper_df's overlapping columns
scraper_df_renamed = scraper_df.rename(columns={col: f"{col}_scraper" for col in overlapping_cols})

# Perform outer merge to keep all rows from both dataframes
merged_final = pd.merge(
    merged_df,
    scraper_df_renamed,
    on='asin',
    how='outer',
    suffixes=('', '_scraper')
)

# For overlapping columns, keep merged_df's values, fill missing with scraper_df's values
for col in overlapping_cols:
    scraper_col = f"{col}_scraper"
    if scraper_col in merged_final.columns:
        # Fill NaN values in merged_df's column with values from scraper_df's column
        merged_final[col] = merged_final[col].fillna(merged_final[scraper_col])
        # Drop the renamed scraper column
        merged_final = merged_final.drop(columns=[scraper_col])

# Final check: Remove any duplicate ASINs that might have been created during merge
final_duplicates = merged_final['asin'].duplicated().sum()
if final_duplicates > 0:
    print(f"\n⚠️  Found {final_duplicates} duplicate ASINs after merge. Removing duplicates (keeping first)...")
    merged_final = merged_final.drop_duplicates(subset=['asin'], keep='first')
    print(f"   After removal: {len(merged_final)} rows")
else:
    print(f"\n✓ No duplicate ASINs in final merged dataset")

# Verify no duplicates remain
assert merged_final['asin'].duplicated().sum() == 0, "ERROR: Duplicate ASINs still exist!"
print(f"✓ Verified: No duplicate ASINs in final dataset")

print(f"\nFinal merged dataset: {len(merged_final)} rows, {len(merged_final.columns)} columns")
print(f"Unique ASINs: {merged_final['asin'].nunique()}")
print(f"\nFirst few rows:")
merged_final.head()


Scraper data: 17375 rows, 51 columns
Merged data: 36879 rows, 69 columns

⚠️  Found 361 duplicate ASINs in merged_df. Removing duplicates (keeping first)...
   After removal: 36518 rows
⚠️  Found 80 duplicate ASINs in scraper_df. Removing duplicates (keeping first)...
   After removal: 17295 rows

Overlapping columns (excluding 'asin'): 20
  Examples: ['sd_parent_asin', 'sd_is_frequently_returned', 'sd_number_bought_past_month', 'sd_previous_price', 'sd_average_rating', 'sd_list_price', 'sd_stars', 'sd_price', 'sd_title', 'sd_availability_status']

✓ No duplicate ASINs in final merged dataset
✓ Verified: No duplicate ASINs in final dataset

Final merged dataset: 53255 rows, 99 columns
Unique ASINs: 53255

First few rows:


,Unnamed: 0,asin,category,query,page,source_section,type,title,image,has_prime,...,n_clusters_sig,color_entropy,largest_cluster_pct,edge_density_z,n_clusters_sig_z,color_entropy_z,bg_white_pct_z,bg_neutral_pct_z,largest_cluster_pct_z,clutter_score
0,8356.0,0060977310,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.0,0.813145,0.540889,-0.040414,0.865509,0.362852,0.257159,-0.780686,0.033760,0.406105
1,1268.0,0062338676,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.0,0.890175,0.402629,0.424844,0.865509,0.841808,-2.434161,-1.130873,-0.823726,1.741266
2,17948.0,0063048701,clothing_shoes_jewelry,watch,5.0,results,search_product,Hands of Time: A Watchmaker’s History,https://m.media-amazon.com/images/I/91VLR0isFS...,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8294.0,0071393900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,0.836369,0.474182,0.984388,-0.169091,0.507254,-2.127693,-1.010499,-0.379951,1.416461
4,8492.0,0072863560,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.0,0.965062,0.336905,-0.172539,0.865509,1.307432,-2.371960,-2.982260,-1.231344,2.001610
